# EII — Phase 1: Grid Configuration Sensitivity Analysis (v2)

Runs EII and Area extraction across all grid configurations required
for the MAUP sensitivity analysis (Paper Outline Section 3.7).

**Configurations covered:**

| Step | Analysis | Grids | Years |
|------|----------|-------|-------|
| 1.1 | Shape (hex vs. square) | HEX-20, SQ-20 | 1985, 2020 |
| 1.2 | Scale sensitivity | HEX-10, HEX-20, HEX-40 | 1985, 2020 |
| 1.3 | Zoning (jitter) | 25 HEX-20 realizations | 1985, 2020 |

**Two metrics extracted per cell:**
- **EII** (`eii_`): proportion of perimeter intercepting natural habitat
  → uses `categorical=True`, `all_touched=True` on perimeter geometries
- **Area** (`area_`): proportion of natural habitat pixels inside the polygon
  → uses `stats=['count','sum']`, `all_touched=False` on polygon interiors

**Why different methods for EII and Area:**
The Area calculation uses `sum/count` on the binary raster (1=natural, 0=non-natural)
because `categorical=True` is sensitive to nodata encoding: if nodata=0 in raster
metadata, non-natural pixels are excluded from the total, producing area=1 everywhere.
`stats=['count','sum']` correctly excludes only true nodata pixels.

---
## 1. Configuration — edit only this cell

In [ ]:
# ============================================================
# CONFIGURATION — edit paths here, run everything else as-is
# ============================================================

# Folder containing the MapBiomas binary rasters (.tif)
RASTER_FOLDER = r"D:\Modelo_LAPIG\rasters_binarios"

# Folder containing the named grid shapefiles
GRID_FOLDER = r"D:\Modelo_LAPIG\grids"

# Folder containing the 25 jitter shapefiles
JITTER_FOLDER = r"D:\Modelo_LAPIG\jitter_grids"

# Root output folder (subfolders created automatically per configuration)
OUTPUT_ROOT = r"D:\Modelo_LAPIG\phase1_sensitivity"

# Years to process in Phase 1
TARGET_YEARS = ["1985", "2020"]

# Nodata fallback (used only if raster metadata has no nodata defined)
NODATA_FALLBACK = 255

# Pixel value representing natural vegetation in the binary rasters
VEGETATION_VALUE = 1

# Scenario label mapping: lowercase fragment of filename → label
# 'reclass' covers files named reclass_YYYY.tif (observed baseline)
SCENARIO_MAP = {
    "reclass": "OBS",
    "obs":     "OBS",
    "tnc1":    "TNC1",
    "tnc2":    "TNC2",
    "bau":     "BAU",
    "gov":     "GOV",
}

# Named grid configurations for steps 1.1 and 1.2
NAMED_GRIDS = {
    "HEX-10": "hex_10000ha.shp",
    "HEX-20": "hex_20000ha.shp",
    "HEX-40": "hex_40000ha.shp",
    "SQ-20":  "sq_20000ha.shp",
}

---
## 2. Dependencies

In [ ]:
import importlib, subprocess, sys

for pkg in ["rasterstats", "geopandas", "rasterio", "pandas", "numpy", "matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    else:
        print(f"OK {pkg}")

---
## 3. Utility functions

In [ ]:
import os
import re
import glob
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

YEAR_RE = re.compile(r"(19|20)\d{2}")


def extract_year(filename: str) -> str:
    m = YEAR_RE.search(filename)
    if not m:
        raise ValueError(f"Year (YYYY) not found in: {filename}")
    return m.group(0)


def infer_scenario(filename: str, scenario_map: dict) -> str:
    lower = filename.lower()
    for key, label in scenario_map.items():
        if key.lower() in lower:
            return label
    return "OBS"  # safe fallback for observed baseline rasters


def ensure_id(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if "ID_UNICO" not in gdf.columns:
        gdf = gdf.copy()
        gdf["ID_UNICO"] = gdf.index.astype(int)
    return gdf


def to_perimeters(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Convert polygon grid to boundary lines (EII transects)."""
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.boundary
    gdf = gdf.explode(index_parts=False).reset_index(drop=True)
    return gdf


def reproject_if_needed(gdf: gpd.GeoDataFrame, raster_path: str) -> gpd.GeoDataFrame:
    with rasterio.open(raster_path) as src:
        crs_raster = src.crs
    if gdf.crs is None:
        raise ValueError("Grid has no CRS defined.")
    if crs_raster and gdf.crs != crs_raster:
        return gdf.to_crs(crs_raster)
    return gdf


def get_nodata(raster_path: str, fallback: int = 255) -> int:
    """Read nodata from raster metadata; use fallback only if not defined."""
    with rasterio.open(raster_path) as src:
        nd = src.nodata
    if nd is None:
        warnings.warn(
            f"No nodata in raster metadata ({os.path.basename(raster_path)}). "
            f"Using fallback={fallback}.")
        return fallback
    return int(nd)


def filter_rasters_by_year(raster_folder: str, target_years: list) -> list:
    all_rasters = sorted(glob.glob(os.path.join(raster_folder, "*.tif")))
    filtered = [
        r for r in all_rasters
        if extract_year(os.path.basename(r)) in target_years
    ]
    if not filtered:
        raise FileNotFoundError(
            f"No rasters found for years {target_years} in {raster_folder}")
    return filtered


print("Utility functions loaded.")

---
## 4. Extraction functions — EII and Area

**EII** uses `categorical=True` + `all_touched=True` on perimeter geometries.

**Area** uses `stats=['count', 'sum']` + `all_touched=False` on polygon interiors.
This correctly handles rasters where nodata=0 in metadata (non-natural pixels
must not be excluded from the denominator).

In [ ]:
# ------------------------------------------------------------------
# EII extraction (perimeter transects — categorical pixel counting)
# ------------------------------------------------------------------

def extract_eii_raster(
    gdf_perimeters: gpd.GeoDataFrame,
    raster_path: str,
    checkpoint_folder: str,
    config_id: str,
) -> pd.DataFrame:
    """
    Extract EII for one raster using perimeter geometries as transects.
    Returns DataFrame: ID_UNICO, px_tot_{tag}, px_veg_{tag}, eii_{tag}
    """
    base = os.path.basename(raster_path)
    year = extract_year(base)
    scenario = infer_scenario(base, SCENARIO_MAP)
    tag = f"{config_id}_{scenario}_{year}"

    checkpoint = os.path.join(checkpoint_folder, f"eii_{tag}.csv")
    if os.path.exists(checkpoint):
        print(f"    [ckpt] eii_{tag}")
        return pd.read_csv(checkpoint)

    print(f"    [run]  eii_{tag}")
    gdf_proj = reproject_if_needed(gdf_perimeters, raster_path)

    # nodata=255: hardcoded regardless of raster metadata.
    # Raster metadata declares nodata=0 (non-vegetation), but 0 is
    # valid data that must be counted in the denominator.
    # True outside-domain pixels have value 255.
    EII_NODATA = 255

    stats = zonal_stats(
        gdf_proj, raster_path,
        categorical=True, nodata=EII_NODATA, all_touched=True,
    )

    ids    = gdf_perimeters["ID_UNICO"].values
    px_veg = np.array([s.get(VEGETATION_VALUE, 0) or 0 for s in stats], dtype=float)
    px_tot = np.array(
        [sum(v for k, v in s.items() if k != EII_NODATA) for s in stats], dtype=float)

    with np.errstate(invalid="ignore", divide="ignore"):
        eii = np.where(px_tot > 0, px_veg / px_tot, np.nan)

    df = pd.DataFrame({
        "ID_UNICO":      ids,
        f"px_tot_{tag}": px_tot.astype(int),
        f"px_veg_{tag}": px_veg.astype(int),
        f"eii_{tag}":    np.round(eii, 4),
    })
    df.to_csv(checkpoint, index=False)
    return df


# ------------------------------------------------------------------
# Area extraction (polygon interiors — sum/count on binary raster)
# ------------------------------------------------------------------

def extract_area_raster(
    gdf_polygons: gpd.GeoDataFrame,
    raster_path: str,
    checkpoint_folder: str,
    config_id: str,
) -> pd.DataFrame:
    """
    Extract within-cell area metric (proportion of natural vegetation).

    Key parameters:
      stats=['count','sum'] with nodata=255
      - count: all pixels where value != 255 (includes 0=non-veg AND 1=veg)
      - sum:   sum of pixel values = count of pixels where value=1 (veg)
      - area:  sum / count

    nodata=255 is hardcoded because raster metadata declares nodata=0,
    which would incorrectly exclude non-vegetation pixels from the
    denominator, producing area=1.0 everywhere.
    True outside-domain pixels have value 255.
    """
    base = os.path.basename(raster_path)
    year = extract_year(base)
    scenario = infer_scenario(base, SCENARIO_MAP)
    tag = f"{config_id}_{scenario}_{year}"

    checkpoint = os.path.join(checkpoint_folder, f"area_{tag}.csv")
    if os.path.exists(checkpoint):
        print(f"    [ckpt] area_{tag}")
        return pd.read_csv(checkpoint)

    print(f"    [run]  area_{tag}")

    # Reproject grid to raster CRS (WGS84) for correct spatial overlay
    gdf_proj = reproject_if_needed(gdf_polygons, raster_path)

    # nodata=255: hardcoded regardless of raster metadata
    # Raster metadata declares nodata=0 (non-vegetation), but 0 is
    # valid data that must be counted in the denominator.
    stats = zonal_stats(
        gdf_proj, raster_path,
        stats=["count", "sum"],
        nodata=255,
        all_touched=False,
    )

    ids    = gdf_polygons["ID_UNICO"].values
    px_tot = np.array([s["count"] or 0 for s in stats], dtype=float)
    px_veg = np.array([s["sum"]   or 0 for s in stats], dtype=float)

    with np.errstate(invalid="ignore", divide="ignore"):
        area = np.where(px_tot > 0, px_veg / px_tot, np.nan)

    df = pd.DataFrame({
        "ID_UNICO":    ids,
        f"area_{tag}": np.round(area, 4),
    })
    df.to_csv(checkpoint, index=False)
    return df


# ------------------------------------------------------------------
# Run both metrics for one full grid configuration
# ------------------------------------------------------------------

def run_config(
    grid_path: str,
    raster_paths: list,
    config_id: str,
    output_root: str,
) -> pd.DataFrame:
    """
    Run EII + Area extraction for one grid configuration across all
    target rasters. Returns consolidated DataFrame.
    Saves result as CSV: output_root/config_id/eii_area_{config_id}.csv
    """
    config_folder     = os.path.join(output_root, config_id)
    checkpoint_folder = os.path.join(config_folder, "checkpoints")
    os.makedirs(checkpoint_folder, exist_ok=True)

    consolidated_path = os.path.join(config_folder, f"eii_area_{config_id}.csv")
    if os.path.exists(consolidated_path):
        print(f"  [done] {config_id} — loading {consolidated_path}")
        return pd.read_csv(consolidated_path)

    print(f"\n{'='*55}")
    print(f"  Config: {config_id} | Grid: {os.path.basename(grid_path)}")
    print(f"{'='*55}")

    gdf       = gpd.read_file(grid_path)
    gdf       = ensure_id(gdf)
    gdf_perim = to_perimeters(gdf)
    print(f"  {len(gdf):,} cells | {len(gdf_perim):,} perimeter features")

    df_result = gdf[["ID_UNICO"]].copy()

    for raster_path in raster_paths:
        # EII — perimeter transects
        df_eii  = extract_eii_raster(gdf_perim, raster_path, checkpoint_folder, config_id)
        df_result = df_result.merge(df_eii, on="ID_UNICO", how="left")

        # Area — polygon interiors
        df_area = extract_area_raster(gdf, raster_path, checkpoint_folder, config_id)
        df_result = df_result.merge(df_area, on="ID_UNICO", how="left")

    df_result.to_csv(consolidated_path, index=False, encoding="utf-8-sig")
    print(f"  Saved: {consolidated_path}")
    return df_result


print("Extraction functions loaded.")

---
## 5. Sanity check — verify area calculation on one raster

Before running the full pipeline, verify that area values show
genuine variation (not all 0 or all 1).

In [ ]:
import matplotlib.pyplot as plt

rasters_phase1 = filter_rasters_by_year(RASTER_FOLDER, TARGET_YEARS)
print(f"Rasters selected: {len(rasters_phase1)}")
for r in rasters_phase1:
    print(f"  {os.path.basename(r)}")

# Quick test on HEX-20 with first raster
test_raster = rasters_phase1[0]
test_grid   = os.path.join(GRID_FOLDER, NAMED_GRIDS["HEX-20"])

gdf_test = gpd.read_file(test_grid)
gdf_test = ensure_id(gdf_test)
gdf_test_proj = reproject_if_needed(gdf_test, test_raster)
nodata_test = get_nodata(test_raster, NODATA_FALLBACK)

print(f"\nRaster nodata value: {nodata_test}")
print(f"Testing area extraction on: {os.path.basename(test_raster)}")

# nodata=255: hardcoded — raster metadata declares nodata=0 (non-veg),
# but 0 is valid data. True outside-domain pixels have value 255.
stats_test = zonal_stats(
    gdf_test_proj.head(200), test_raster,
    stats=["count", "sum"], nodata=255, all_touched=False
)

px_tot_t = np.array([s["count"] or 0 for s in stats_test], dtype=float)
px_veg_t = np.array([s["sum"]   or 0 for s in stats_test], dtype=float)

with np.errstate(invalid="ignore", divide="ignore"):
    area_t = np.where(px_tot_t > 0, px_veg_t / px_tot_t, np.nan)

print(f"Area — min: {np.nanmin(area_t):.4f} | max: {np.nanmax(area_t):.4f} | "
      f"mean: {np.nanmean(area_t):.4f}")

if np.nanmax(area_t) == np.nanmin(area_t):
    print("WARNING: all area values are identical — check raster nodata encoding.")
else:
    print("OK: area values show genuine variation.")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(area_t[np.isfinite(area_t)], bins=30, color="steelblue", edgecolor="white")
ax.set_xlabel("Area (proportion natural vegetation)")
ax.set_ylabel("Cell count")
ax.set_title(f"Area distribution check — HEX-20 / {os.path.basename(test_raster)}")
plt.tight_layout()
plt.show()

---
## 6. Steps 1.1 and 1.2 — Named grid configurations

Runs EII + Area for HEX-10, HEX-20, HEX-40, SQ-20.

In [ ]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)
results_named = {}

for config_id, shp_name in NAMED_GRIDS.items():
    grid_path = os.path.join(GRID_FOLDER, shp_name)
    if not os.path.exists(grid_path):
        print(f"WARNING: grid not found — {grid_path}. Skipping {config_id}.")
        continue
    df = run_config(grid_path, rasters_phase1, config_id, OUTPUT_ROOT)
    results_named[config_id] = df
    # Quick check
    area_cols = [c for c in df.columns if c.startswith("area_")]
    for col in area_cols:
        vals = df[col].dropna()
        print(f"  {col}: min={vals.min():.4f} mean={vals.mean():.4f} max={vals.max():.4f}")

print(f"\nSteps 1.1 and 1.2 complete: {list(results_named.keys())}")

---
## 7. Step 1.3 — Jitter realizations (25 grids)

Runs EII + Area for all 25 displaced HEX-20 grids.

In [ ]:
jitter_shapefiles = sorted(
    glob.glob(os.path.join(JITTER_FOLDER, "hex20_jitter_*.shp"))
)
print(f"Jitter shapefiles found: {len(jitter_shapefiles)} (expected: 25)")

results_jitter = {}

for shp_path in jitter_shapefiles:
    stem      = os.path.splitext(os.path.basename(shp_path))[0]
    suffix    = stem.replace("hex20_jitter_", "")
    config_id = f"JITTER_{suffix}"
    df = run_config(shp_path, rasters_phase1, config_id, OUTPUT_ROOT)
    results_jitter[config_id] = df

print(f"\nStep 1.3 complete. Jitter realizations processed: {len(results_jitter)}")

---
## 8. Compute 5×5 Area × EII frequency matrices

Joint frequency matrices are the primary output for the MAUP analysis.

In [ ]:
BIN_EDGES  = np.array([0, 0.2, 0.4, 0.6, 0.8, 1.0])
BIN_LABELS = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]


def compute_matrix(df: pd.DataFrame, config_id: str, year: str):
    """
    Compute 5x5 Area x EII frequency matrix (as proportions).
    Returns (matrix, n_valid_cells) or (None, 0) if columns not found.
    """
    eii_cols  = [c for c in df.columns if c.startswith(f"eii_{config_id}_")  and year in c]
    area_cols = [c for c in df.columns if c.startswith(f"area_{config_id}_") and year in c]

    if not eii_cols:
        print(f"  MISSING eii column  : {config_id} / {year}")
        return None, 0
    if not area_cols:
        print(f"  MISSING area column : {config_id} / {year}")
        return None, 0

    eii  = df[eii_cols[0]].values.astype(float)
    area = df[area_cols[0]].values.astype(float)

    mask = np.isfinite(eii) & np.isfinite(area)
    eii  = np.clip(eii[mask],  0, 1 - 1e-9)
    area = np.clip(area[mask], 0, 1 - 1e-9)

    eii_bin  = np.clip(np.digitize(eii,  BIN_EDGES, right=False) - 1, 0, 4)
    area_bin = np.clip(np.digitize(area, BIN_EDGES, right=False) - 1, 0, 4)

    counts = np.zeros((5, 5), dtype=int)
    for ab, eb in zip(area_bin, eii_bin):
        counts[ab, eb] += 1

    return counts / counts.sum(), mask.sum()


# Named configurations
all_matrices = {}
for config_id, df in results_named.items():
    all_matrices[config_id] = {}
    for year in TARGET_YEARS:
        mat, n = compute_matrix(df, config_id, year)
        if mat is not None:
            all_matrices[config_id][year] = mat
            print(f"  Matrix OK: {config_id} / {year} (n={n:,} cells)")

# Jitter realizations
jitter_matrices = {year: [] for year in TARGET_YEARS}
for year in TARGET_YEARS:
    for config_id, df in results_jitter.items():
        mat, n = compute_matrix(df, config_id, year)
        if mat is not None:
            jitter_matrices[year].append(mat)
    print(f"  Jitter matrices: {year} — {len(jitter_matrices[year])}/25")

print("\nAll matrices computed.")

---
## 9. Jitter stability statistics

Mean, SD, and CV of the 5×5 matrix across 25 realizations.

In [ ]:
jitter_summary_folder = os.path.join(OUTPUT_ROOT, "jitter_summary")
os.makedirs(jitter_summary_folder, exist_ok=True)

for year in TARGET_YEARS:
    mats = jitter_matrices.get(year, [])
    if len(mats) < 2:
        print(f"Not enough jitter matrices for {year}.")
        continue

    stack    = np.stack(mats, axis=0)   # (25, 5, 5)
    mean_mat = stack.mean(axis=0)
    sd_mat   = stack.std(axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        cv_mat = np.where(mean_mat > 0, sd_mat / mean_mat, np.nan)

    for name, mat in [("mean", mean_mat), ("sd", sd_mat), ("cv", cv_mat)]:
        pd.DataFrame(
            np.round(mat, 6), index=BIN_LABELS, columns=BIN_LABELS
        ).to_csv(
            os.path.join(jitter_summary_folder, f"jitter_{name}_{year}.csv"),
            encoding="utf-8-sig"
        )

    mean_cv = np.nanmean(cv_mat)
    max_cv  = np.nanmax(cv_mat)
    print(f"\nJitter stability — {year} ({len(mats)} realizations):")
    print(f"  Mean CV: {mean_cv:.4f} | Max CV: {max_cv:.4f}")
    print(f"  Interpretation: CV < 0.05 = high stability")

print(f"\nSaved to: {jitter_summary_folder}")

---
## 10. Shape comparison: HEX-20 vs. SQ-20

In [ ]:
shape_folder = os.path.join(OUTPUT_ROOT, "shape_comparison")
os.makedirs(shape_folder, exist_ok=True)

for year in TARGET_YEARS:
    hex20_mat = all_matrices.get("HEX-20", {}).get(year)
    sq20_mat  = all_matrices.get("SQ-20",  {}).get(year)
    if hex20_mat is None or sq20_mat is None:
        print(f"Missing matrix for {year}. Run cells 6-8 first.")
        continue

    diff_mat = hex20_mat - sq20_mat

    # CV of EII per shape
    for config_id, df in [("HEX-20", results_named["HEX-20"]),
                           ("SQ-20",  results_named["SQ-20"])]:
        eii_cols = [c for c in df.columns
                    if c.startswith(f"eii_{config_id}_") and year in c]
        if eii_cols:
            eii = df[eii_cols[0]].dropna()
            cv  = eii.std() / eii.mean()
            print(f"  CV of EII — {config_id} ({year}): {cv:.4f}")

    # Save difference matrix
    pd.DataFrame(
        np.round(diff_mat, 6), index=BIN_LABELS, columns=BIN_LABELS
    ).to_csv(
        os.path.join(shape_folder, f"diff_HEX20_minus_SQ20_{year}.csv"),
        encoding="utf-8-sig")

    # Figure
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, mat, title, cmap, vmin, vmax in [
        (axes[0], hex20_mat, f"HEX-20 ({year})",       "YlOrRd", 0, hex20_mat.max()),
        (axes[1], sq20_mat,  f"SQ-20 ({year})",        "YlOrRd", 0, hex20_mat.max()),
        (axes[2], diff_mat,  "Difference (HEX − SQ)",  "RdBu_r",
         -abs(diff_mat).max(), abs(diff_mat).max()),
    ]:
        im = ax.imshow(mat, origin="lower", aspect="auto",
                       cmap=cmap, vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_xticks(range(5)); ax.set_xticklabels(BIN_LABELS, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(5)); ax.set_yticklabels(BIN_LABELS, fontsize=8)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("EII class", fontsize=9)
        ax.set_ylabel("Area class", fontsize=9)
        for i in range(5):
            for j in range(5):
                ax.text(j, i, f"{mat[i,j]:.3f}", ha="center", va="center", fontsize=7)

    plt.suptitle(f"Shape comparison: HEX-20 vs. SQ-20 — {year}", fontsize=11)
    plt.tight_layout()
    fig_path = os.path.join(shape_folder, f"shape_comparison_{year}.png")
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Figure saved: {fig_path}")

---
## 11. Scale comparison: HEX-10 / HEX-20 / HEX-40

In [ ]:
scale_folder  = os.path.join(OUTPUT_ROOT, "scale_comparison")
os.makedirs(scale_folder, exist_ok=True)

scale_configs = ["HEX-10", "HEX-20", "HEX-40"]
scale_labels  = ["10,000 ha", "20,000 ha", "40,000 ha"]

for year in TARGET_YEARS:
    print(f"\nScale comparison — {year}:")

    cv_values = {}
    for config_id in scale_configs:
        df = results_named.get(config_id)
        if df is None:
            continue
        eii_cols = [c for c in df.columns
                    if c.startswith(f"eii_{config_id}_") and year in c]
        if eii_cols:
            eii = df[eii_cols[0]].dropna()
            cv  = eii.std() / eii.mean()
            cv_values[config_id] = cv
            print(f"  CV of EII — {config_id}: {cv:.4f}")

    mats_available = [
        (cfg, lbl, all_matrices.get(cfg, {}).get(year))
        for cfg, lbl in zip(scale_configs, scale_labels)
        if all_matrices.get(cfg, {}).get(year) is not None
    ]
    if not mats_available:
        print(f"  No matrices for {year}.")
        continue

    vmax = max(m.max() for _, _, m in mats_available)
    fig, axes = plt.subplots(1, len(mats_available), figsize=(5 * len(mats_available), 4))
    if len(mats_available) == 1:
        axes = [axes]

    for ax, (cfg, lbl, mat) in zip(axes, mats_available):
        im = ax.imshow(mat, origin="lower", aspect="auto",
                       cmap="YlOrRd", vmin=0, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_xticks(range(5)); ax.set_xticklabels(BIN_LABELS, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(5)); ax.set_yticklabels(BIN_LABELS, fontsize=8)
        cv_str = f" | CV={cv_values[cfg]:.3f}" if cfg in cv_values else ""
        ax.set_title(f"{cfg} — {lbl}{cv_str}", fontsize=9)
        ax.set_xlabel("EII class", fontsize=9)
        ax.set_ylabel("Area class", fontsize=9)
        for i in range(5):
            for j in range(5):
                ax.text(j, i, f"{mat[i,j]:.3f}",
                        ha="center", va="center", fontsize=7)

    plt.suptitle(f"Scale sensitivity: HEX-10 / HEX-20 / HEX-40 — {year}", fontsize=11)
    plt.tight_layout()
    fig_path = os.path.join(scale_folder, f"scale_comparison_{year}.png")
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Figure saved: {fig_path}")

---
## 12. Phase 1 summary report

In [ ]:
rows = []

for config_id, df in {**results_named, **results_jitter}.items():
    analysis = "jitter" if config_id.startswith("JITTER") else "shape/scale"
    for year in TARGET_YEARS:
        eii_cols  = [c for c in df.columns if "eii_"  in c and year in c]
        area_cols = [c for c in df.columns if "area_" in c and year in c]
        if not eii_cols or not area_cols:
            continue
        eii  = df[eii_cols[0]].dropna()
        area = df[area_cols[0]].dropna()
        rows.append({
            "analysis":   analysis,
            "config_id":  config_id,
            "year":       year,
            "n_cells":    len(eii),
            "eii_mean":   round(eii.mean(), 4),
            "eii_sd":     round(eii.std(), 4),
            "eii_cv":     round(eii.std() / eii.mean(), 4),
            "area_mean":  round(area.mean(), 4),
            "area_sd":    round(area.std(), 4),
        })

df_summary = pd.DataFrame(rows)
summary_path = os.path.join(OUTPUT_ROOT, "phase1_summary.csv")
df_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Phase 1 Summary (named configurations):")
named_mask = df_summary["analysis"] == "shape/scale"
print(df_summary[named_mask].to_string(index=False))
print(f"\nFull summary (including jitter) saved to: {summary_path}")